<a href="https://colab.research.google.com/github/veniola/DX703-Assignments/blob/main/Homework_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework 9: Text Classification with Fine-Tuned BERT

### Due: Midnight on November 1st (with 2-hour grace period) — Worth 85 points

In this final homework, we’ll explore **fine-tuning pretrained Transformer models** for text classification using the **IMDB Movie Review** dataset. You’ll begin by training and evaluating a DistilBERT baseline, then conduct experiments to investigate how training configurations, dataset size, and model architecture affect predictive performance and computational cost.

You’ll complete three problems:

* **Problem 1:** Compare three context-length training configurations and their validation performance and training times. You’ll also compare your best configuration with the initial DistilBERT baseline.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **DistilBERT and BERT-base** to analyze the trade-offs between model size, accuracy, and training time.

In each problem, you’ll report experimental results, interpret the differences, and justify a choice based on validation performance and computational cost.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are three problems with six graded questions in total. Each graded question is worth 14 points, with one additional point for completing the homework.


### Developing with Increasing Amounts of Data

Fine-tuning Transformer models can be computationally expensive. In this assignment, you will use a **progressive development workflow**: first verify that your code works with a small dataset, then increase the amount of training data as your experiments become more reliable.

The global `DATA_MODE` setting controls the training subset used for initial model development:

- `"smoke"` — 1% of training data, for checking that your code runs.
- `"dev"` — 10% of training data, for debugging and development.
- `"baseline"` — 25% of training data, for establishing a meaningful baseline.
- `"full"` — 100% of the training pool, when full-data training is needed.

The subsets are reproducible and nested: larger subsets contain the examples used in smaller subsets.

**Important:** The development setting affects only the training data. The validation and test sets remain fixed, so evaluation results are comparable. Do not use the test set to select models or tune hyperparameters.

The graded experiments in Problems 1–3 specify their own training-data fractions, independently of `DATA_MODE`. Changing this global setting does not change the datasets used in those experiments.

For the baseline, use `DATA_MODE = "baseline"`. For the graded experiments, follow the settings specified in each problem.


In [1]:
# Install once per new Colab runtime
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate

# ---- Backend BEFORE importing keras ----
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel
from sklearn.model_selection import train_test_split

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.32.0 which is incompatible.


### Here is where you can set global hyperparameters for this homework

In [2]:

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 128
EPOCHS = 2
BATCH = 32
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "baseline"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")


Data mode: baseline
Training fraction: 25%


### Load and Preprocess the IMDB Movie Review Dataset

In [3]:

# ------------ Load and Partition IMDB ------------

imdb = load_dataset("stanfordnlp/imdb")

# Preserve the original held-out test set
test_ds = imdb["test"]

# Create a fixed, stratified validation set
indices = np.arange(len(imdb["train"]))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=np.array(imdb["train"]["label"])
)

train_pool = imdb["train"].select(train_idx.tolist())
val_ds = imdb["train"].select(val_idx.tolist())

# ------------ Reproducible Nested Training Subsets ------------

# Shuffle each class independently, then interleave classes.
# This gives balanced, nested training subsets.
rng = np.random.default_rng(SEED)

labels = np.array(train_pool["label"])
neg_idx = np.flatnonzero(labels == 0)
pos_idx = np.flatnonzero(labels == 1)

rng.shuffle(neg_idx)
rng.shuffle(pos_idx)

train_order = np.column_stack((neg_idx, pos_idx)).ravel()

def get_training_subset(frac):
    """Return a balanced, nested fraction of the training pool."""
    if not 0 < frac <= 1:
        raise ValueError("frac must be in (0, 1].")

    n_pairs = max(1, round(len(train_order) * frac / 2))
    selected = train_order[:2 * n_pairs]

    return train_pool.select(selected.tolist())

# Apply the development throttle
train_ds = get_training_subset(TRAIN_FRAC)

# ------------ NumPy Arrays for Keras ------------

X_tr = np.array(train_ds["text"], dtype=object)
y_tr = np.array(train_ds["label"], dtype="int32")

X_va = np.array(val_ds["text"], dtype=object)
y_va = np.array(val_ds["label"], dtype="int32")

X_te = np.array(test_ds["text"], dtype=object)
y_te = np.array(test_ds["label"], dtype="int32")

# ------------ Summary ------------

def _counts(ds):
    values = np.array(ds["label"], dtype=int)
    return len(values), np.bincount(values, minlength=2).tolist()

print(f"Data mode: {DATA_MODE} ({TRAIN_FRAC:.0%})")
print("Training pool:", _counts(train_pool))
print("Training subset:", _counts(train_ds))
print("Validation:", _counts(val_ds))
print("Test:", _counts(test_ds))


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Data mode: baseline (25%)
Training pool: (22500, [11250, 11250])
Training subset: (5624, [2812, 2812])
Validation: (2500, [1250, 1250])
Test: (25000, [12500, 12500])


### Build and train a baseline Distil-Bert Text Classifier

In [4]:

# ---- DistilBERT baseline: develop using the selected data throttle ----

print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {MAX_LEN}")
print(f"Batch size: {BATCH}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 128
Batch size: 32


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a grain dataset. The grain dataset is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 210s 866ms/step - acc: 0.7642 - loss: 0.4945 - val_acc: 0.8460 - val_loss: 0.3564
Epoch 2/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 101s 579ms/step - acc: 0.8608 - loss: 0.3282 - val_acc: 0.8464 - val_loss: 0.3634



Best epoch: 1
Validation loss (restored model): 0.3564
Validation accuracy: 0.8460
Validation F1: 0.8462

Validation confusion matrix:
[[1056  194]
 [ 191 1059]]

Training time: 00:05:16



# Problem 1 — Mini Sweep: Context Length and Training Cost (3 Runs)

In this problem, we'll investigate how **context length** (`MAX_LEN`) affects the performance and computational cost of fine-tuning DistilBERT.

A longer context allows the model to use more of each text, potentially improving its predictions. However, longer sequences require more computation and memory, particularly because of the attention mechanism.

We'll compare three configurations, using different batch sizes and learning rates appropriate for each context length.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25` — use 25% of the training dataset.
* `EPOCHS = 2` — maximum number of training epochs.
* **EarlyStopping** with `restore_best_weights=True`.
* Same random seed for all runs.
* Same training and validation examples for all runs (don't reshuffle or resplit between runs).

## Run these 3 configurations

| MAX_LEN | Learning rate | Batch size |
|---|---|---|
| 128 | `2e-5` | 32 |
| 256 | `1e-5` | 16 |
| 512 | `7.5e-6` | 8 |

These configurations represent progressively longer contexts, with smaller batch sizes to accommodate the increased memory requirements.

**Important:** Because learning rate and batch size also change, this experiment compares three practical training configurations. It does not isolate the effect of context length alone.

For each run, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Identify the configuration with the **lowest validation loss**. Problems 2 and 3 use their own fixed configurations to isolate the effects being studied.

## If you encounter an Out of Memory error

* At `MAX_LEN = 256`, reduce `BATCH` from 16 to 8.
* At `MAX_LEN = 512`, reduce `BATCH` from 8 to 4.

If you make such a change, record it when discussing your results.

Then answer the graded questions.


In [6]:
# =========================================================
# Problem 1A: Mini Sweep - Context Length & training Cost
# =========================================================

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 128
EPOCHS = 2
BATCH = 32
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "dev"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model_1A = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model_1A.compile(
    optimizer=keras.optimizers.Adam(2e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model_1A.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model_1A.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model_1A.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))

Data mode: dev
Training fraction: 10%


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a grain dataset. The grain dataset is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 194s 822ms/step - acc: 0.7765 - loss: 0.4643 - val_acc: 0.8524 - val_loss: 0.3364
Epoch 2/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 134s 767ms/step - acc: 0.8750 - loss: 0.2974 - val_acc: 0.8464 - val_loss: 0.3631



Best epoch: 1
Validation loss (restored model): 0.3364
Validation accuracy: 0.8524
Validation F1: 0.8510

Validation confusion matrix:
[[1077  173]
 [ 196 1054]]

Training time: 00:05:34


In [7]:
# =========================================================
# Problem 1B: Mini Sweep - Context Length & training Cost
# =========================================================

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 256
EPOCHS = 2
BATCH = 16
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "dev"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model_1B = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model_1B.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model_1B.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model_1B.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model_1B.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))

Data mode: dev
Training fraction: 10%


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a grain dataset. The grain dataset is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 202s 441ms/step - acc: 0.8318 - loss: 0.3846 - val_acc: 0.8720 - val_loss: 0.3106
Epoch 2/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 136s 386ms/step - acc: 0.9147 - loss: 0.2261 - val_acc: 0.8884 - val_loss: 0.2935



Best epoch: 2
Validation loss (restored model): 0.2935
Validation accuracy: 0.8884
Validation F1: 0.8883

Validation confusion matrix:
[[1112  138]
 [ 141 1109]]

Training time: 00:05:41


In [8]:
# =========================================================
# Problem 1C: Mini Sweep - Context Length & training Cost
# =========================================================

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 512
EPOCHS = 2
BATCH = 8
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "dev"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model_1A = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model_1A.compile(
    optimizer=keras.optimizers.Adam(7.5e-6),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model_1A.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model_1A.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model_1A.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))

Data mode: dev
Training fraction: 10%


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a grain dataset. The grain dataset is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/2
703/703 ━━━━━━━━━━━━━━━━━━━━ 185s 201ms/step - acc: 0.8483 - loss: 0.3590 - val_acc: 0.9080 - val_loss: 0.2457
Epoch 2/2
703/703 ━━━━━━━━━━━━━━━━━━━━ 135s 192ms/step - acc: 0.9202 - loss: 0.2057 - val_acc: 0.8992 - val_loss: 0.2742



Best epoch: 1
Validation loss (restored model): 0.2457
Validation accuracy: 0.9076
Validation F1: 0.9076

Validation confusion matrix:
[[1135  115]
 [ 116 1134]]

Training time: 00:05:41


### Graded Questions

In [9]:
# Set a1a to the validation accuracy at the epoch with minimum
# validation loss for the configuration with the lowest validation loss.

a1a = 0.2457                                       # Replace 0.0 with your answer

In [10]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.2457


### Question a1b:

* How did validation accuracy change across the three context lengths (128 → 256 → 512)? Were the improvements equally large?
* How did training time change as context length increased?
* Compare the best Problem 1 configuration with the initial DistilBERT baseline. How much did validation accuracy improve, and what additional training time was required?
* Which configuration would you choose, considering both validation performance and computational cost? Explain your reasoning.

**Note:** Because learning rate and batch size also changed, you cannot attribute the differences in performance solely to context length.


In [ ]:
a1b = """
Validation accuracy across all three context lengths did not change drastically.
"""



In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1b = {a1b}')

# Problem 2 — How Much Data Is Enough? (3 Runs)

In this problem, you'll investigate how the amount of training data affects the performance and computational cost of fine-tuning DistilBERT.

More training data generally improves predictive performance, but also increases training time. We want to determine whether the additional improvement is worth the computational cost.

## Setup

* Use a fixed DistilBERT configuration: `MAX_LEN = 256`, `LR = 1e-5`, and `BATCH = 16`, independently of the selected configuration in Problem 1.
* Keep the validation dataset fixed across all runs.
* Use `EPOCHS = 2` and EarlyStopping with `restore_best_weights=True`.
* Use the same random seed for all runs.

## Run these 3 configurations

| SUBSET_FRAC | Training data |
|---|---|
| 0.25 | 25% of the training set |
| 0.50 | 50% of the training set |
| 1.00 | 100% of the training set |

For each run, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Select the configuration with the lowest validation loss.

Then answer the graded questions.



In [ ]:
# ========================================================
# Problem 2A: How Much Data is Enough? (SUBSET_FRAC=0.25)
# ========================================================

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 256
EPOCHS = 2
BATCH = 16
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "baseline"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model_1A = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model_1A.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model_1A.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model_1A.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model_1A.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))

### Graded Questions

In [ ]:
# Set a2a to the validation accuracy at the epoch with minimum
# validation loss for the configuration with the lowest validation loss.

a2a = 0.0  # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

### Question a2b:

* How did validation accuracy change as the training dataset increased from 25% to 50% to 100%? Were the improvements equally large?
* How did training time change as the dataset increased?
* Which dataset size would you choose, considering both predictive performance and computational cost? Explain your reasoning using your results.


In [ ]:
a2b = """
Your answer here
"""



In [ ]:
## Graded Answer
# DO NOT change this cell in any way

print(f'a2b = {a2b}')

# Problem 3 — Model Swap: Speed vs. Accuracy (2 Runs)

In this problem, we'll compare two pretrained Transformer models with different capacities:

* **DistilBERT** — a smaller, computationally efficient model.
* **BERT-base** — a larger model with greater capacity, but higher computational cost.

Both models can be fine-tuned for the same text classification task. Our goal is to determine whether the larger model's improvement in predictive performance justifies its additional training time.

## Setup

Use the following settings for both models:

* `MAX_LEN = 256`
* `LR = 1e-5`
* `BATCH = 16`
* `SUBSET_FRAC = 0.50`
* `EPOCHS = 2`
* EarlyStopping with `restore_best_weights=True`
* Same random seed
* Same training and validation examples

Only the model architecture changes.

## Run these 2 configurations

| Model | Pretrained preset |
|---|---|
| DistilBERT | `distil_bert_base_en_uncased` |
| BERT-base | `bert_base_en_uncased` |

## How to switch models

**DistilBERT:**

```python
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)
```

**BERT-base:**

```python
preproc = kh.models.BertTextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased", num_classes=2, preprocessor=preproc
)
```

## What to do

Train and evaluate both models using the same experimental settings.

For each model, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Compare the results, then answer the graded questions.

**Note:** This experiment compares models under the same training budget, not necessarily at their individually optimal hyperparameters.


In [ ]:
# Your code here; add as many cells as you wish



### Graded Questions

In [ ]:
# Set a3a to the validation accuracy at the epoch with minimum
# validation loss for the model with the lowest validation loss.

a3a = 0.0  # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

### Question a3b:

* Which model achieved the higher validation accuracy? How large was the difference?
* How did their total training times compare?
* Given limited computational resources, which model would you choose? Explain whether the improvement in predictive performance justifies the additional computational cost.


In [ ]:
a3b = """
Your answer here
"""



In [ ]:
## Graded Answer
# DO NOT change this cell in any way

print(f'a3b = {a3b}')